# Task h - Part 2
Select one target for decarbonisation (i.e., one CO2 allowance limit). What is the (global/system-wide) CO2 price required to achieve that decarbonisation level? Search for information on the existing CO2 tax in your countries (if any) and discuss your results. Is the model in agreement with the existing CO2 tax (either national CO2 tax and/or the European CO2 price coming from the ETS)? Why or why not?

In [101]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.options.mode.string_storage = "python"

In [102]:
# 1. Load datasets
data_solar = pd.read_csv('data/pv_optimal.csv', sep=';', index_col=0, parse_dates=True)
data_wind = pd.read_csv('data/onshore_wind_1979-2017.csv', sep=';', index_col=0, parse_dates=True)
data_el = pd.read_csv('data/electricity_demand.csv', sep=';', index_col=0, parse_dates=True)

In [103]:
# 2. Setup and Process Costs 
year_cost = 2030
url = f"https://raw.githubusercontent.com/PyPSA/technology-data/v0.11.0/outputs/costs_{year_cost}.csv"
costs = pd.read_csv(url, index_col=[0, 1])

costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3
defaults = {"FOM": 0, "VOM": 0, "efficiency": 1, "fuel": 0, "investment": 0, "lifetime": 25, "discount rate": 0.07}
costs = costs.value.unstack().fillna(defaults)
costs.at["CCGT", "fuel"] = costs.at["gas", "fuel"]

def annuity(r, n):
    return r / (1.0 - 1.0 / (1.0 + r) ** n)

costs["marginal_cost"] = costs["VOM"] + costs["fuel"] / costs["efficiency"]
ann = costs.apply(lambda x: annuity(x["discount rate"], x["lifetime"]), axis=1)
costs["capital_cost"] = (ann + costs["FOM"] / 100) * costs["investment"]


In [104]:
# 3. Initialize Network
n = pypsa.Network()
snapshots = pd.date_range("2011-01-01 00:00", "2011-12-31 23:00", freq="h")
n.set_snapshots(snapshots)

In [105]:
# 4. Carriers with CO2 Emission Factors
n.add("Carrier", "coal", co2_emissions=0.336, color="indianred")
n.add("Carrier", "CCGT", co2_emissions=0.198, color="yellow-green")
n.add("Carrier", "onwind", co2_emissions=0, color="dodgerblue")
n.add("Carrier", "solar", co2_emissions=0, color="gold")
n.add("Carrier", "nuclear", co2_emissions=0, color="orange")

In [106]:
# 5. Add Buses (Spain + France + Portugal + Morocco)
n.add("Bus", "Spain electricity", x=-3.7038, y=40.4168)
n.add("Bus", "FR", x=2.2137,  y=46.2276)
n.add("Bus", "PT", x=-8.2245, y=39.3999)
n.add("Bus", "MA", x=-7.0926, y=31.7917)

In [107]:
# 6. Add Interconnectors
lines = [
    ("ES-FR", "Spain electricity", "FR", 2800),
    ("ES-PT", "Spain electricity", "PT", 3000),
    ("FR-PT", "FR",                "PT", 2200),
    ("ES-MA", "Spain electricity", "MA", 900),
]

for name, bus0, bus1, s_nom in lines:
    n.add("Line", name, bus0=bus0, bus1=bus1, s_nom=s_nom, 
          s_nom_extendable=False, x=0.1, v_nom=400)


In [108]:
# 7. Add Domestic Generators (Spain)
n.add("Generator", "coal", bus="Spain electricity", carrier="coal",
      p_nom_extendable=True, p_nom_max=11700,
      capital_cost=costs.at["coal", "capital_cost"], 
      marginal_cost=costs.at["coal", "marginal_cost"])

n.add("Generator", "CCGT", bus="Spain electricity", carrier="CCGT",
      p_nom_extendable=True, p_nom_max=25300,
      capital_cost=costs.at["CCGT", "capital_cost"],
      marginal_cost=costs.at["CCGT", "marginal_cost"])

n.add("Generator", "onwind", bus="Spain electricity", carrier="onwind",
      p_nom_extendable=True, capital_cost=costs.at["onwind", "capital_cost"],
      p_max_pu=data_wind['ESP'].iloc[:8760].values)

n.add("Generator", "solar", bus="Spain electricity", carrier="solar",
      p_nom_extendable=True, capital_cost=costs.at["solar", "capital_cost"],
      p_max_pu=data_solar['ESP'].iloc[:8760].values)

In [109]:
# 8. Add Neighboring Generators
neighbor_gens = {
    "FR": [("nuclear_FR", "nuclear", 63000, 5.0), ("wind_FR", "onwind", 6600, 1.4)],
    "PT": [("wind_PT", "onwind", 4378, 1.4), ("gas_PT", "CCGT", 3800, 46.8)],
    "MA": [("thermal_MA", "CCGT", 4771, 46.8), ("wind_MA", "onwind", 292, 1.4)],
}

for country, gens in neighbor_gens.items():
    for name, carrier, p_nom, mc in gens:
        n.add("Generator", name, bus=country, carrier=carrier, 
              p_nom=p_nom, p_nom_extendable=False, marginal_cost=mc)


In [110]:
# 9. Add Loads
n.add("Load", "demand_ESP", bus="Spain electricity", p_set=data_el['ESP'].iloc[:8760].values)
n.add("Load", "demand_FR", bus="FR", p_set=468e6/8760) # Avg hourly demand
n.add("Load", "demand_PT", bus="PT", p_set=48e6/8760)
n.add("Load", "demand_MA", bus="MA", p_set=28e6/8760)

In [111]:
# 10. Interconnected CO2 Limit

n.add("GlobalConstraint", "CO2Limit", 
      carrier_attribute="co2_emissions", 
      sense="<=", 
      constant=20e6) 


In [112]:
# Solve
n.optimize(solver_name='highs')

# Results extraction
co2_price = abs(n.global_constraints.at["CO2Limit", "mu"])
print(f"--- Interconnected Model Results (Task 2h) ---")
print(f"Required CO2 Price to hit limit: {co2_price:.2f} EUR/tCO2")
print(f"Total Interconnected System Cost: {n.objective / 1e9:.2f} Billion EUR")

C:\Users\20221122\AppData\Local\Temp\ipykernel_21156\96760857.py:2: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize(solver_name='highs')
Index(['Spain electricity', 'FR', 'PT', 'MA'], dtype='str', name='name')
Index(['ES-FR', 'ES-PT', 'FR-PT', 'ES-MA'], dtype='str', name='name')
Index(['ES-FR', 'ES-PT', 'FR-PT', 'ES-MA'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 3/3 [00:00<00:00, 104.04it/s]
INFO:linopy.io: Writing time: 0.29s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 122644 primals, 289087 duals
Objective: 1.68e+10
Solver model: available
Solver message: Optimal

IN

--- Interconnected Model Results (Task 2h) ---
Required CO2 Price to hit limit: 56.09 EUR/tCO2
Total Interconnected System Cost: 16.84 Billion EUR
